In [7]:
import numpy as np
import pandas as pd
import warnings

from tqdm import tqdm, TqdmWarning
from datetime import datetime
from pathlib import Path
from typing import Literal


warnings.filterwarnings("ignore", category=TqdmWarning)

In [8]:
Type = Literal["train", "test"]
Split = Literal["split_01", "split_02", "split_03", "split_04", "split_05", "split_06", "split_07", "split_08", "split_09", "split_10", "split_11", "split_12", "split_13", "split_14", "split_15", "split_16", "split_17", "split_18", "split_19", "split_20"]


EPS = np.finfo(float).eps


def now() -> str:
    return datetime.now().astimezone().strftime("%Y%m%d-%H%M%S-%z")

### Data Loading

In [9]:
def load_log_df(type: Type, **kwargs) -> pd.DataFrame:
    return pd.read_parquet(f"../artifacts/kaggle/{type}_log.parquet", **kwargs)

def load_flc_df(type: Type, split: Split, **kwargs) -> pd.DataFrame:
    return pd.read_parquet(f"../artifacts/kaggle/{split}/{type}_flc.parquet", **kwargs)


def __ingest_dfs(type: Type):
    log_df = pd.read_csv(f"../artifacts/kaggle/{type}_log.csv", index_col="object_id")
    log_df.to_parquet(f"../artifacts/kaggle/{type}_log.parquet")

    splits = sorted(log_df["split"].unique())
    for split in splits:
        flc_df = pd.read_csv(f"../artifacts/kaggle/{split}/{type}_full_lightcurves.csv")
        flc_df.to_parquet(f"../artifacts/kaggle/{split}/{type}_flc.parquet")


__ingest_dfs(type="train")
__ingest_dfs(type="test")

### Feature Engineering

In [10]:
def load_feats_df(type: Type, split: Split, **kwargs):
    return pd.read_parquet(f"../artifacts/feats/{split}/{type}_feats.parquet", **kwargs)


def __ingest_feats(type: Type):
    log_df = load_log_df(type=type)

    with tqdm(desc="Ingesting features", total=len(log_df["split"].unique()), unit="split") as pb:
        for split, log_sub_df in log_df.groupby("split"):
            feats_buf = []

            with tqdm(desc=f"Ingesting features for `{split}`", total=len(log_sub_df), unit="obj") as sub_pb:
                for obj_id, log_row in log_sub_df.iterrows():
                    flc_sub_df = load_flc_df(type=type, split=split, filters=[("object_id", "==", obj_id)]) # type: ignore
                    feats = __load_feats_for_obj(log_row, flc_sub_df)
                    feats_buf += feats

                    sub_pb.update()

            Path(f"../artifacts/feats/{split}").mkdir(parents=True, exist_ok=True)
            pd.DataFrame(feats_buf).to_parquet(f"../artifacts/feats/{split}/{type}_feats.parquet")

            pb.update()


def __load_feats_for_obj(log_row: pd.Series, flc_sub_df: pd.DataFrame) -> dict:
    flc_sub_df["Flux_ratio"] = flc_sub_df["Flux"] / flc_sub_df["Flux_err"]
    # pivot_flc_sub_df = flc_sub_df.pivot_table(index="Time (MJD)", columns="Filter", values=["Flux", "Flux_err", "Flux_ratio"])

    feats = log_row.to_dict()

    # def q25(series: pd.Series): return series.quantile(0.25)
    # def q75(series: pd.Series): return series.quantile(0.75)
    # def skew(series: pd.Series): return series.skew()
    # def kurt(series: pd.Series): return series.kurt()

    # for agg in [np.mean, np.std, np.min, np.max, np.median, q25, q75, skew, kurt]:
    #     for feat in ["Flux", "Flux_err", "Flux_ratio"]:
    #         feats[f"{feat}_{agg.__name__}"] = agg(flc_sub_df[feat])
            
    #         for filter in ["u", "g", "r", "i", "z", "y"]:
    #             feats[f"{feat}_{agg.__name__}_{filter}"] = agg(pivot_flc_sub_df[(feat, filter)]) if filter in pivot_flc_sub_df.columns else np.nan

    feats[f"Flux_mean"] = flc_sub_df["Flux"].mean()

    return feats


__ingest_feats(type="train")
# __ingest_feats(type="test")


# def de_extinct(log_df: pd.DataFrame, flc_df: pd.DataFrame):
#     EXTINCTION_COEFFS = {
#         "u": 4.81,
#         "g": 3.64,
#         "r": 2.70,
#         "i": 2.06,
#         "z": 1.58,
#         "y": 1.31
#     }

#     ebv = flc_df["object_id"].map(log_df["EBV"])
#     r_λ = flc_df["Filter"].map(EXTINCTION_COEFFS)

#     c_λ = np.pow(10, 0.4 * r_λ * ebv)

#     flc_df["Flux"] *= c_λ
#     flc_df["Flux_err"] *= c_λ


# TODO Consider extinction.fitzpatrick99

Ingesting features: 100%|██████████| 20/20 [00:09<00:00,  2.03split/s]


In [11]:
feats = load_feats_df(type="train", split="split_01")

feats

,0
0,Z
1,Z_err
2,EBV
3,SpecType
4,English Translation
...,...
1235,SpecType
1236,English Translation
1237,split
1238,target


### Training

In [12]:
# __dirpath = Path("../artifacts/predictions")
# __dirpath.mkdir(parents=True, exist_ok=True)

# prediction_df.to_csv(f"{__dirpath}/submission-{now()}.csv", index=False)